In [3]:
from pathlib import Path
import ast

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


ROOT = Path.cwd().parent

FEATURE_FILE = ROOT / "data" / "features" / "movies_features.csv"

df = pd.read_csv(FEATURE_FILE)

df.head()

,movie_id,title,original_title,overview,tagline,release_date,runtime,original_language,genres,keywords,...,belongs_to_collection,poster_path,backdrop_path,homepage,release_year,release_month,release_decade,genre_count,keyword_count,runtime_category
0,969681,Spider-Man: Brand New Day,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,A brand new day starts now.,2026-07-29,145.0,en,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...","[{'id': 9678, 'name': 'mind control'}, {'id': ...",...,"{'id': 531241, 'name': 'Spider-Man (MCU) Colle...",/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg,/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg,https://spidermanbrandnewday.movie,2026.0,7.0,2020.0,3,19,Long
1,1423191,Resident Evil,Resident Evil,Medical courier Bryan unwittingly finds himsel...,A new era of evil.,2026-09-16,95.0,en,"[{'id': 27, 'name': 'Horror'}, {'id': 878, 'na...","[{'id': 9744, 'name': 'outbreak'}, {'id': 1034...",...,NaN,/i7UyjfPio0VFHB9rBUZSFyhOoM8.jpg,/3icyRAqgakNcQn6aDVz9libFmBA.jpg,https://residentevil.movie,2026.0,9.0,2020.0,3,20,Medium
2,1368337,The Odyssey,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",Defy the gods.,2026-07-15,173.0,en,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'id': 3799, 'name': 'ship'}, {'id': 3265, 'n...",...,NaN,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,/bulFtfy3oBQtCnAMSi7g6DSSds3.jpg,https://www.universalpictures.com/movies/the-o...,2026.0,7.0,2020.0,3,18,Very Long
3,1101383,The End of Oak Street,The End of Oak Street,After a mysterious cosmic event rips Oak Stree...,Where goes the neighborhood.,2026-08-12,100.0,en,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...","[{'id': 3822, 'name': 'teleportation'}, {'id':...",...,NaN,/fYXqpgPmHMphSF2W30GbTeJVIa5.jpg,/b9q9VmbXDvJmTziRqkwdEmFdwhr.jpg,https://www.endofoakstreet.com,2026.0,8.0,2020.0,3,17,Medium
4,1032863,The Love Hypothesis,The Love Hypothesis,"Olive Smith, a biology PhD candidate, and Dr. ...",Fake dating. Real chemistry.,2026-09-23,112.0,en,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'id': 818, 'name': 'based on novel or book'}...",...,"{'id': 1783817, 'name': 'The Love Hypothesis C...",/vfZxVHextAGC70zrNhS8lsROqP1.jpg,/o7Oy9Gbx1CCyaweL8xUhtMW4Puq.jpg,https://www.amazon.com/dp/B0H4WZS7VX,2026.0,9.0,2020.0,2,11,Medium


In [4]:
df["genres"] = df["genres"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

df["keywords"] = df["keywords"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [5]:
def extract_names(items):
    return " ".join(
        item["name"]
        for item in items
        if isinstance(item, dict) and "name" in item
    )

In [6]:
df["genres_text"] = df["genres"].apply(extract_names)
df["keywords_text"] = df["keywords"].apply(extract_names)

In [7]:
df["text"] = (
    df["title"].fillna("") + " "
    + df["overview"].fillna("") + " "
    + df["genres_text"].fillna("") + " "
    + df["keywords_text"].fillna("")
)

In [8]:
df[["title", "text"]].head()

,title,text
0,Spider-Man: Brand New Day,Spider-Man: Brand New Day Fighting crime full-...
1,Resident Evil,Resident Evil Medical courier Bryan unwittingl...
2,The Odyssey,"The Odyssey Odysseus, the legendary King of It..."
3,The End of Oak Street,The End of Oak Street After a mysterious cosmi...
4,The Love Hypothesis,"The Love Hypothesis Olive Smith, a biology PhD..."


In [9]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

In [10]:
tfidf_matrix = tfidf.fit_transform(df["text"])

In [11]:
print("TF-IDF matrix shape:")
print(tfidf_matrix.shape)

TF-IDF matrix shape:
(2953, 5000)


In [12]:
feature_names = tfidf.get_feature_names_out()

print("Number of features:", len(feature_names))
print(feature_names[:20])

Number of features: 5000
['000' '10' '101' '11' '12' '13' '13th' '14' '1453' '14th' '15' '15th'
 '17' '17th' '18' '1890s' '18th' '19' '1910s' '1920s']


In [13]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [14]:
print(similarity_matrix.shape)

(2953, 2953)


In [15]:
similarity_matrix[0][0]

np.float64(1.0)

In [16]:
def get_similar_movies(movie_title, n=10):
    matches = df[
        df["title"].str.lower() == movie_title.lower()
    ]

    if matches.empty:
        return pd.DataFrame()

    movie_index = matches.index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[movie_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similar_indices = [
        index
        for index, score in similarity_scores[1:n + 1]
    ]

    result = df.loc[
        similar_indices,
        ["title", "vote_average", "popularity"]
    ].copy()

    result["similarity"] = [
        similarity_matrix[movie_index][index]
        for index in similar_indices
    ]

    return result

In [17]:
get_similar_movies("Spider-Man: Brand New Day")

,title,vote_average,popularity,similarity
142,Spider-Man,7.400,43.0777,0.477849
93,Spider-Man: Across the Spider-Verse,8.351,53.2974,0.474922
76,Spider-Man: No Way Home,7.947,59.1728,0.474722
182,Spider-Man: Homecoming,7.333,37.6345,0.468931
261,The Amazing Spider-Man,6.800,32.5195,0.453940
430,The Amazing Spider-Man 2,6.600,26.0606,0.433206
185,Spider-Man: Into the Spider-Verse,8.401,40.0776,0.414286
272,Spider-Man 3,6.500,31.5585,0.381748
1295,Madame Web,5.265,15.2782,0.375418
1550,Spider-Man 2,7.300,13.9774,0.358267


In [18]:
import joblib

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    tfidf,
    MODEL_DIR / "tfidf_vectorizer.joblib"
)

['/home/elgmouri/projects/moviesHub/models/tfidf_vectorizer.joblib']

In [19]:
joblib.dump(
    tfidf_matrix,
    MODEL_DIR / "tfidf_matrix.joblib"
)

['/home/elgmouri/projects/moviesHub/models/tfidf_matrix.joblib']

In [20]:
print("Movies:", len(df))
print("TF-IDF matrix:", tfidf_matrix.shape)
print("Similarity matrix:", similarity_matrix.shape)

Movies: 2953
TF-IDF matrix: (2953, 5000)
Similarity matrix: (2953, 2953)
